In [78]:
import requests
import json
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
# List of emotion nodes' URLs
emotion_nodes = {
    'anger': 'http://localhost:8001/predict',
    'joy': 'http://localhost:8002/predict',
    'sadness': 'http://localhost:8003/predict',
    'surprise': 'http://localhost:8004/predict',
    'fear': 'http://localhost:8005/predict'
}

In [80]:
import pandas as pd

In [ ]:
# Load the testing dataset
file_path = r'C:\Users\HP\Desktop\major project\multi_class_emotion_300_dataset.csv'
test_data = pd.read_csv(file_path)

In [ ]:
# Confidence threshold to consider a secondary emotion
CONFIDENCE_THRESHOLD = 0.1

In [83]:
# Function to send request to the emotion node and get the response
def get_prediction(emotion, text):
    try:
        response = requests.post(emotion_nodes[emotion], json={'text': text})
        if response.status_code == 200:
            return response.json()
        else:
            raise Exception(f"Error in {emotion} node: {response.text}")
    except Exception as e:
        print(f"Error in {emotion} node: {e}")
        return {"prediction": "Not Emotion", "confidence": 0.0}


In [84]:
# Function to determine the primary and secondary emotions
def determine_emotions(predictions):
    # Get the primary emotion (highest confidence)
    primary_emotion = max(predictions, key=lambda x: x['confidence'])
    
    # Get secondary emotions that are close in confidence to the primary emotion
    secondary_emotions = [
        emotion for emotion in predictions 
        if abs(emotion['confidence'] - primary_emotion['confidence']) <= CONFIDENCE_THRESHOLD and emotion != primary_emotion
    ]
    
    return primary_emotion, secondary_emotions

In [ ]:
# Function to process the dataset and evaluate the performance
def evaluate_model(test_data):
    true_labels = []
    predicted_labels = []
    secondary_confidences = []  # To capture secondary emotion confidences
    
    for index, row in test_data.iterrows():
        text = row['Sentence']
        true_label = row['Emotion'].strip().lower()  # Normalize true label to lowercase
        true_labels.append(true_label)
        
        # Get predictions from all emotion nodes
        predictions = []
        for emotion in emotion_nodes.keys():
            prediction = get_prediction(emotion, text)
            predictions.append({
                'emotion': emotion,
                'prediction': prediction['prediction'].strip().lower(),  # Normalize to lowercase
                'confidence': prediction['confidence']
            })
        
        # Determine the primary and secondary emotions
        primary_emotion, secondary_emotions = determine_emotions(predictions)
        
        # For accuracy calculation
        predicted_labels.append(primary_emotion['emotion'])
        secondary_confidences.append({se['emotion']: se['confidence'] for se in secondary_emotions})
        
        # Optionally, print or log predictions for each sentence
        print(f"Text: {text}")
        print(f"True Label: {true_label}")
        print(f"Primary Emotion: {primary_emotion['emotion']} (Confidence: {primary_emotion['confidence']:.2f})")
        #print(f"Secondary Emotions: {[{'emotion': se['emotion'], 'confidence': se['confidence']:.2f} for se in secondary_emotions]}")
        secondary_emotions_formatted = [
            {'emotion': se['emotion'], 'confidence': f"{se['confidence']:.2f}"} for se in secondary_emotions
        ]
        print(f"Secondary Emotions: {secondary_emotions_formatted}")
        print('-' * 50)
    
    # Compute the evaluation metrics
    correct_predictions = 0
    for i, true_label in enumerate(true_labels):
        primary_correct = predicted_labels[i] == true_label
        secondary_correct = any(se == true_label for se in secondary_confidences[i].keys())
        if primary_correct or secondary_correct:
            correct_predictions += 1

    accuracy = correct_predictions / len(true_labels)
    
    return accuracy

In [ ]:
# Ensure the 'label' column matches the emotion labels you are working with
accuracy = evaluate_model(test_data)

print(f"Accuracy: {accuracy:.4f}")

Text: I felt a deep sense of loss after the breakup.
True Label: sadness
Primary Emotion: fear (Confidence: 0.98)
Secondary Emotions: [{'emotion': 'sadness', 'confidence': '0.96'}]
--------------------------------------------------
Text: The thought of failing the exam kept me awake with dread.
True Label: fear
Primary Emotion: anger (Confidence: 0.99)
Secondary Emotions: [{'emotion': 'sadness', 'confidence': '0.97'}, {'emotion': 'fear', 'confidence': '0.99'}]
--------------------------------------------------
Text: He was terrified when He saw the shadow outside my window.
True Label: fear
Primary Emotion: anger (Confidence: 0.99)
Secondary Emotions: [{'emotion': 'joy', 'confidence': '0.94'}, {'emotion': 'fear', 'confidence': '0.98'}]
--------------------------------------------------
Text: We was shocked to see the unexpected guest at my party.
True Label: surprise
Primary Emotion: surprise (Confidence: 0.99)
Secondary Emotions: [{'emotion': 'anger', 'confidence': '0.92'}, {'emotion'